In [1]:
!pip install boto3 botocore --quiet

In [2]:
import os
import boto3
import botocore

aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

session = boto3.session.Session(aws_access_key_id=aws_access_key_id,
                                aws_secret_access_key=aws_secret_access_key)

s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name)

bucket = s3_resource.Bucket(bucket_name)

#upload the model directory without git
def upload_directory_to_s3(local_directory, s3_prefix):
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            if ".git" in relative_path:
                continue
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"{file_path} -> {s3_key}")
            bucket.upload_file(file_path, s3_key)


def list_objects(prefix):
    filter = bucket.objects.filter(Prefix=prefix)
    for obj in filter.all():
        print(obj.key)

In [3]:
from dotenv import load_dotenv

local_base_dir = os.path.expanduser("~/shared")
params_file = f"{local_base_dir}/output-1.env"

load_dotenv(params_file)

True

In [4]:
model_name = os.environ.get("MODEL_NAME")
local_path = os.environ.get("TUNED_MODEL_LOCATION")

tuned_model_suffix = "tuned"
s3_path = f"{model_name}-{tuned_model_suffix}"

upload_directory_to_s3(str(local_path), s3_path)

/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/tokenizer_config.json -> meta-llama/Llama-3.2-1B-Instruct-tuned/tokenizer_config.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/model.safetensors -> meta-llama/Llama-3.2-1B-Instruct-tuned/model.safetensors
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/config.json -> meta-llama/Llama-3.2-1B-Instruct-tuned/config.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/generation_config.json -> meta-llama/Llama-3.2-1B-Instruct-tuned/generation_config.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/special_tokens_map.json -> meta-llama/Llama-3.2-1B-Instruct-tuned/special_tokens_map.json
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/chat_template.jinja -> meta-llama/Llama-3.2-1B-Instruct-tuned/chat_template.jinja
/opt/app-root/src/shared/meta-llama/Llama-3.2-1B-Instruct/merged_model/tokenizer.jso

In [5]:
list_objects(s3_path)

meta-llama/Llama-3.2-1B-Instruct-tuned/chat_template.jinja
meta-llama/Llama-3.2-1B-Instruct-tuned/config.json
meta-llama/Llama-3.2-1B-Instruct-tuned/generation_config.json
meta-llama/Llama-3.2-1B-Instruct-tuned/model.safetensors
meta-llama/Llama-3.2-1B-Instruct-tuned/special_tokens_map.json
meta-llama/Llama-3.2-1B-Instruct-tuned/tokenizer.json
meta-llama/Llama-3.2-1B-Instruct-tuned/tokenizer_config.json


In [6]:
from dotenv import set_key, dotenv_values
from pathlib import Path

params_file = f"{local_base_dir}/output-2.env"
Path(params_file).write_text("")

set_key(params_file, "MODEL_PATH", s3_path)

!cat {params_file}

MODEL_PATH='meta-llama/Llama-3.2-1B-Instruct-tuned'
